# Imports, config, session and utilities

In [ ]:
import json
from pathlib import Path
import sys


current_file_path = Path().resolve()
sys.path.insert(0, str(current_file_path.parent))
# These import statement after adding parent file path to recognize src python files
from src.support_agent.config import get_settings
from src.support_agent.snowflake_client import create_snowpark_session


# Getting .env config and init snowflake session
settings = get_settings()
session = create_snowpark_session(settings)

# Main code

In [ ]:
query = """ SELECT * FROM PROJECT_DB.EVAL.RESULTS """

results_df = session.sql(query).to_pandas()
results_df["total_output_tokens"] = (
    results_df["total_output_tokens"]
    .apply(json.loads)
    .apply(lambda d: d["total"])
)
results_df.shape

In [ ]:
results_df.columns

In [ ]:
print("\n" + "=" * 80)
print("CHECK REASONING ")
print("=" * 80)
print(
    f"Total tickets found with similar problems in vector database: {results_df.used_search.value_counts()}"
)

In [ ]:
# Print summary statistics
print("\n" + "=" * 80)
print("EVALUATION SUMMARY")
print("=" * 80)
print(f"Total tickets evaluated: {len(results_df)}")
print("\nAverage Scores:")
print(f"  Faithfulness:      {results_df['faithfulness_score'].mean():.3f}")
print(f"  Answer Relevancy:  {results_df['answer_relevancy_score'].mean():.3f}")
print(
    f"  Context Precision: {results_df['context_precision_score'].mean():.3f}"
)
print(f"  Context Recall:    {results_df['context_recall_score'].mean():.3f}")
print(f"  Overall Average:   {results_df['average_score'].mean():.3f}")

if "language" in results_df.columns:
    print(r"\Average score by Language:")
    print(results_df.groupby("language")["average_score"].mean())

if "priority" in results_df.columns:
    print(r"\Average score by Priority:")
    print(results_df.groupby("priority")["average_score"].mean())

In [ ]:
# Print summary statistics
print("\n" + "=" * 80)
print("USAGE SUMMARY")
print("=" * 80)
print(
    f"  Input tokens usage:      {results_df['total_input_tokens'].mean():.3f}"
)
print(f"  Output tokens usage:  {results_df['total_output_tokens'].mean():.3f}")


if "language" in results_df.columns:
    print("Average score by Language:")
    print(results_df.groupby("language")["total_input_tokens"].mean())

if "priority" in results_df.columns:
    print("Average score by Priority:")
    print(results_df.groupby("priority")["total_output_tokens"].mean())

# Close session 

In [ ]:
session.close()